The purpose of this notebook is to tests the different ucb bounds in the rcps code in order to obtain plots like the ones in figure 4 of the RCPS paper.

In [1]:
import sys
import math
import numpy as np
import pandas as pd

sys.path.insert(0, "/home/joseba.dalmau/puncc")

from deel.puncc.api.RCPS import (clt_ucb,
                                 simplified_hoeffding_ucb,
                                 tighter_hoeffding_ucb,
                                 bentkus_ucb,
                                 hoeffding_bentkus_ucb,
                                 bernstein_ucb,
                                 wsr_ucb,
)

In [2]:
DIST_SPECS = [
    ("Ber(mu)", None),
    ("Beta(0.1, .)", 0.1),
    ("Beta(1, . )", 1.0),
    ("Beta(10, .)", 10.0),
]

MUS = [0.1, 0.01, 0.001]

NS = [int(math.floor(10 ** r)) for r in [2, 2.5, 3, 3.5, 4]]

METHODS = ["Bernstein", "CLT", "HB", "sim-Hoef", "WSR"]

In [3]:
def draw_samples(rng, beta_a, mu, rows, n):
    """Draw loss samples with mean mu."""
    if beta_a is None:
        return rng.binomial(1, mu, size=(rows, n)).astype(float)

    beta_b = beta_a * (1.0 / mu - 1.0)
    return rng.beta(beta_a, beta_b, size=(rows, n))


def effective_batch_size(n, requested_batch, max_elements=4_000_000):
    """
    Keep memory moderate for large n.
    For n=10000, this gives at most 400 rows per batch by default.
    """
    return max(1, min(requested_batch, max_elements // n))

In [4]:
def simulate_one_setting(rng, dist_label, beta_a, mu, n, delta, reps, batch):
    gaps = {method: np.empty(reps, dtype=float) for method in METHODS}
    covered = {method: 0 for method in METHODS}

    pos = 0
    batch = effective_batch_size(n, batch)

    while pos < reps:
        rows = min(batch, reps - pos)

        losses = draw_samples(rng, beta_a, mu, rows, n)
        emp_risks = losses.mean(axis=1)
        risk_sd = losses.std(axis=1, ddof=1)

        ucbs = {
    "sim-Hoef": np.array([
        simplified_hoeffding_ucb(lambda lam, r=r: r, delta, n)(0.0)
        for r in emp_risks
    ]),
    "tight-Hoef": np.array([
        tighter_hoeffding_ucb(lambda lam, r=r: r, delta, n)(0.0)
        for r in emp_risks
    ]),
    "Bentkus": np.array([
        bentkus_ucb(lambda lam, r=r: r, delta, n)(0.0)
        for r in emp_risks
    ]),
    "HB": np.array([
        hoeffding_bentkus_ucb(lambda lam, r=r: r, delta, n)(0.0)
        for r in emp_risks
    ]),
    "Bernstein": np.array([
        bernstein_ucb(lambda lam, r=r: r, lambda lam, s=s: s, delta, n)(0.0)
        for r, s in zip(emp_risks, risk_sd)
    ]),
    "CLT": np.array([
        clt_ucb(lambda lam, r=r: r, lambda lam, s=s: s, delta, n)(0.0)
        for r, s in zip(emp_risks, risk_sd)
    ]),
    "WSR": np.array([
        wsr_ucb([lambda lam, l=l: l for l in row], delta)(0.0)
        for row in losses
    ]),
}

        for method, ucb in ucbs.items():
            covered[method] += int(np.sum(ucb >= mu))
            gaps[method][pos:pos + rows] = ucb - mu

        pos += rows

    records = []
    for method in METHODS:
        records.append(
            {
                "dist": dist_label,
                "mu": mu,
                "n": n,
                "method": method,
                "coverage": covered[method] / reps,
                "median_gap": float(np.median(gaps[method])),
            }
        )

    return records


In [5]:
def run_simulation(delta, reps, batch, seed):
    rng = np.random.default_rng(seed)
    all_records = []

    total = len(DIST_SPECS) * len(MUS) * len(NS)
    done = 0

    for dist_label, beta_a in DIST_SPECS:
        for mu in MUS:
            for n in NS:
                done += 1
                print(
                    f"[{done:02d}/{total}] dist={dist_label:12s} "
                    f"mu={mu:g} n={n} reps={reps}"
                )
                all_records.extend(
                    simulate_one_setting(
                        rng=rng,
                        dist_label=dist_label,
                        beta_a=beta_a,
                        mu=mu,
                        n=n,
                        delta=delta,
                        reps=reps,
                        batch=batch,
                    )
                )

    return pd.DataFrame(all_records)

In [6]:
# Plotting
def plot_panel_grid(df, y_col, ylabel, outpath, delta, include_clt=True):
    methods = ["Bernstein", "CLT", "HB", "sim-Hoef", "WSR"]
    if not include_clt:
        methods = ["Bernstein", "HB", "sim-Hoef", "WSR"]

    colors = {
        "Bernstein": "black",
        "CLT": "teal",
        "HB": "blue",
        "sim-Hoef": "goldenrod",
        "WSR": "red",
    }
    linestyles = {
        "Bernstein": (0, (1, 3)),
        "CLT": (0, (3, 2, 1, 2)),
        "HB": (0, (5, 3)),
        "sim-Hoef": (0, (3, 1, 1, 1)),
        "WSR": "-",
    }

    fig, axes = plt.subplots(
        nrows=len(DIST_SPECS),
        ncols=len(MUS),
        figsize=(10.5, 8.0),
        sharex=True,
    )

    for r, (dist_label, _) in enumerate(DIST_SPECS):
        for c, mu in enumerate(MUS):
            ax = axes[r, c]

            for method in methods:
                sub = df[
                    (df["dist"] == dist_label)
                    & (df["mu"] == mu)
                    & (df["method"] == method)
                ].sort_values("n")

                ax.plot(
                    sub["n"],
                    sub[y_col],
                    label=method,
                    color=colors[method],
                    linestyle=linestyles[method],
                    linewidth=1.7,
                )

            ax.set_xscale("log", base=10)
            ax.set_xlim(90, 11000)

            if y_col == "coverage":
                ax.axhline(1.0 - delta, color="black", linewidth=0.8)
                if dist_label == "Ber(mu)":
                    ax.set_ylim(0.20, 1.02)
                elif dist_label == "Beta(0.1, .)":
                    ax.set_ylim(0.78, 1.02)
                elif dist_label == "Beta(1, .)":
                    ax.set_ylim(0.88, 1.02)
                else:
                    ax.set_ylim(0.92, 1.02)
            else:
                ax.set_yscale("log", base=10)
                ax.set_ylim(3e-4, 2e-1)

            if r == 0:
                ax.set_title(f"mu = {mu:g}")

            if c == len(MUS) - 1:
                ax.text(
                    1.04,
                    0.5,
                    dist_label,
                    transform=ax.transAxes,
                    rotation=-90,
                    va="center",
                    ha="left",
                )

            if r == len(DIST_SPECS) - 1:
                ax.set_xlabel("n")

            if c == 0:
                ax.set_ylabel(ylabel)

            ax.grid(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(methods),
        frameon=False,
    )

    fig.tight_layout(rect=[0, 0.07, 1, 1])
    fig.savefig(outpath, dpi=300)
    plt.close(fig)


In [7]:
# Notebook driver cell for Figure 4
from pathlib import Path
from IPython.display import Image, display

# -----------------------------
# Parameters
# -----------------------------

delta = 0.1

# Start small while debugging.
# For a paper-scale run, use reps = 1_000_000.
reps = 2_000

batch = 2_048
seed = 123
outdir = Path("fig4_out")
outdir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Run simulation
# -----------------------------

df = run_simulation(
    delta=delta,
    reps=reps,
    batch=batch,
    seed=seed,
)

# Inspect results in the notebook
display(df.head())
display(df.tail())

# -----------------------------
# Save summary CSV
# -----------------------------

csv_path = outdir / "figure4_simulation_summary.csv"
df.to_csv(csv_path, index=False)

# -----------------------------
# Make plots
# -----------------------------

coverage_path = outdir / "fig4_coverage.png"
gap_path = outdir / "fig4_gap.png"

plot_panel_grid(
    df=df,
    y_col="coverage",
    ylabel="Coverage",
    outpath=coverage_path,
    delta=delta,
    include_clt=True,
)

plot_panel_grid(
    df=df,
    y_col="median_gap",
    ylabel=r"$\hat{R}^{+} - R$",
    outpath=gap_path,
    delta=delta,
    include_clt=False,
)

# -----------------------------
# Display plots inline
# -----------------------------

display(Image(filename=str(coverage_path)))
display(Image(filename=str(gap_path)))

print(f"Saved CSV to: {csv_path}")
print(f"Saved coverage plot to: {coverage_path}")
print(f"Saved gap plot to: {gap_path}")

[01/60] dist=Ber(mu)      mu=0.1 n=100 reps=2000


KeyError: 'tight-Hoef'